# Batch Normalization 实战示例

本notebook演示 Batch Normalization（批归一化）在神经网络训练中的作用：

1. **不使用 BN** vs **使用 BN** 的效果对比
2. 可视化训练过程中 loss 与 accuracy 的变化
3. 展示 BN 层内部参数（γ, β, μ, σ）是如何随训练演化的

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 检查设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## 1. 数据准备：生成非线性分类数据

In [ ]:
# 生成非线性可分的二分类数据（Swiss Roll）
def generate_swiss_roll(n_samples=2000, noise=0.3):
    t = 1.5 * np.pi * (1 + 2 * np.random.rand(n_samples))
    y = 0.3 * np.random.randn(n_samples)
    r = t / np.pi
    x1 = r * np.cos(t) + noise * np.random.randn(n_samples)
    x2 = r * np.sin(t) + noise * np.random.randn(n_samples)
    x3 = y + noise * np.random.randn(n_samples)
    X = np.column_stack([x1, x2, x3])
    # 两类标签：内侧样本 vs 外侧样本
    labels = (r > 1.6).astype(int)
    return X, labels


X, y = generate_swiss_roll(n_samples=3000, noise=0.4)
X = (X - X.mean(axis=0)) / X.std(axis=0)  # 简单标准化

X_train = torch.FloatTensor(X[:2000]).to(device)
y_train = torch.FloatTensor(y[:2000]).unsqueeze(1).to(device)
X_test  = torch.FloatTensor(X[2000:]).to(device)
y_test  = torch.FloatTensor(y[2000:]).unsqueeze(1).to(device)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=256)

# 可视化 3D 数据
fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(X[y==0,0], X[y==0,1], X[y==0,2], c='dodgerblue', alpha=0.5, s=10, label='Class 0')
ax1.scatter(X[y==1,0], X[y==1,1], X[y==1,2], c='crimson',    alpha=0.5, s=10, label='Class 1')
ax1.set_title('Swiss Roll Data (3D)')
ax1.legend()

ax2 = fig.add_subplot(122)
ax2.scatter(X[y==0,0], X[y==0,1], c='dodgerblue', alpha=0.5, s=10, label='Class 0')
ax2.scatter(X[y==1,0], X[y==1,1], c='crimson',    alpha=0.5, s=10, label='Class 1')
ax2.set_title('Swiss Roll Data (x1 vs x2)')
ax2.legend()
plt.tight_layout()
plt.show()

---
## 2. 定义两种网络结构

- **Net without BN**: 普通 5 层全连接网络（relu 激活）
- **Net with BN**: 同结构，但在每个 Linear 层后加入 `BatchNorm1d`

In [ ]:
class NetWithoutBN(nn.Module):
    def __init__(self, input_dim=3, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, 1)
        )
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.1)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        return self.net(x)


class NetWithBN(nn.Module):
    def __init__(self, input_dim=3, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, 1)
        )
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.1)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        return self.net(x)


# 打印网络结构
model_no_bn = NetWithoutBN().to(device)
model_bn    = NetWithBN().to(device)

total_params = lambda m: sum(p.numel() for p in m.parameters())
print(f"Model without BN  — params: {total_params(model_no_bn):,}")
print(f"Model with    BN  — params: {total_params(model_bn):,}  (+ BN's γ/β)")

---
## 3. 训练函数

In [ ]:
def train_model(model, train_loader, test_loader, epochs=80, lr=0.05, device=device):
    model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
    
    history = {'train_loss': [], 'test_acc': []}
    bn_stats = []  # 用于记录 BN 层统计量
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * xb.size(0)
        
        scheduler.step()
        
        # 记录训练 loss
        train_loss = epoch_loss / len(train_loader.dataset)
        history['train_loss'].append(train_loss)
        
        # 记录 BN 统计量（仅对 BN 模型）
        if isinstance(model, NetWithBN) and epoch % 5 == 0:
            gamma_vals, beta_vals, mu_vals, sigma_vals = [], [], [], []
            for m in model.modules():
                if isinstance(m, nn.BatchNorm1d):
                    gamma_vals.append(m.weight.data.cpu().numpy())
                    beta_vals.append(m.bias.data.cpu().numpy())
                    mu_vals.append(m.running_mean.cpu().numpy())
                    sigma_vals.append(np.sqrt(m.running_var.cpu().numpy()))
            bn_stats.append({
                'epoch': epoch,
                'gamma': np.concatenate(gamma_vals),
                'beta':  np.concatenate(beta_vals),
                'mu':    np.concatenate(mu_vals),
                'sigma': np.concatenate(sigma_vals),
            })
        
        # 记录测试准确率
        model.eval()
        correct = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = (torch.sigmoid(model(xb)) > 0.5).float()
                correct += (preds == yb).sum().item()
        test_acc = correct / len(test_loader.dataset)
        history['test_acc'].append(test_acc)
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs}  |  Loss: {train_loss:.4f}  |  Test Acc: {test_acc:.4f}")
    
    return history, bn_stats


---
## 4. 开始训练对比

In [ ]:
print("=" * 60)
print("Training WITHOUT Batch Normalization ...")
print("=" * 60)
model_no_bn = NetWithoutBN().to(device)
hist_no_bn, _ = train_model(model_no_bn, train_loader, test_loader, epochs=80, lr=0.05)

print()
print("=" * 60)
print("Training WITH Batch Normalization ...")
print("=" * 60)
model_bn, hist_bn = train_model(NetWithBN().to(device), train_loader, test_loader, epochs=80, lr=0.2)


---
## 5. 结果可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图：训练 Loss ---
axes[0].plot(hist_no_bn['train_loss'], label='Without BN', color='crimson',  lw=2)
axes[0].plot(hist_bn['train_loss'],    label='With BN',    color='dodgerblue', lw=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('BCE Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# --- 右图：测试 Accuracy ---
axes[1].plot(hist_no_bn['test_acc'], label='Without BN', color='crimson',  lw=2)
axes[1].plot(hist_bn['test_acc'],    label='With BN',    color='dodgerblue', lw=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Test Accuracy', fontsize=12)
axes[1].set_title('Test Accuracy', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0.4, 1.02)

plt.suptitle('Batch Normalization Effect Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nFinal Test Accuracy — Without BN: {hist_no_bn['test_acc'][-1]:.4f}")
print(f"Final Test Accuracy — With    BN: {hist_bn['test_acc'][-1]:.4f}")

---
## 6. Batch Normalization 内部参数演化

In [ ]:
# 提取 BN 参数演化历史
epochs_bn = [s['epoch'] for s in hist_bn]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

axes[0].plot(epochs_bn, [s['gamma'].mean() for s in hist_bn], 'o-', color='steelblue')
axes[0].set_title('γ (scale)  Mean', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_bn, [s['beta'].mean()  for s in hist_bn], 'o-', color='tomato')
axes[1].set_title('β (shift)   Mean', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_bn, [s['mu'].mean()    for s in hist_bn], 'o-', color='green')
axes[2].set_title('EMA μ (running mean)', fontsize=12)
axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)

axes[3].plot(epochs_bn, [s['sigma'].mean() for s in hist_bn], 'o-', color='purple')
axes[3].set_title('EMA σ (running std)', fontsize=12)
axes[3].set_xlabel('Epoch')
axes[3].grid(True, alpha=0.3)

plt.suptitle('Evolution of BatchNorm Parameters (γ, β, μ, σ)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n说明：")
print("  γ 和 β 是可学习参数，网络通过反向传播自动调整它们。")
print("  μ 和 σ 是 EMA 估计的全局均值/标准差，在训练时用于 inference。")

---
## 7. 决策边界可视化

In [ ]:
def plot_decision_boundary(model, X, y, ax, title):
    model.eval()
    x1_range = np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 100)
    x2_range = np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 100)
    xx, yy = np.meshgrid(x1_range, x2_range)
    grid = np.column_stack([xx.ravel(), yy.ravel(), np.zeros(xx.ravel().shape)])
    with torch.no_grad():
        logits = model(torch.FloatTensor(grid).to(device))
        probs = torch.sigmoid(logits).cpu().numpy().reshape(xx.shape)
    ax.contourf(xx, yy, probs, levels=np.linspace(0, 1, 11), cmap='RdBu', alpha=0.6)
    ax.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=1.5)
    ax.scatter(X[y==0,0], X[y==0,1], c='dodgerblue', alpha=0.4, s=8)
    ax.scatter(X[y==1,0], X[y==1,1], c='crimson',    alpha=0.4, s=8)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_decision_boundary(model_no_bn, X, y, axes[0], 'Decision Boundary (No BN)')
plot_decision_boundary(model_bn,    X, y, axes[1], 'Decision Boundary (With BN)')
plt.suptitle('Decision Boundary Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. 总结

从以上实验可以看出：

| 对比维度 | Without BN | With BN |
|---|---|---|
| **收敛速度** | 较慢（loss 下降慢） | 更快（可使用更大的学习率） |
| **最终精度** | 较低 | 更高 |
| **学习率敏感性** | 高（需仔细调参） | 更鲁棒 |
| **梯度流** | 可能梯度消失/爆炸 | 梯度更稳定 |

Batch Normalization 的核心思想是在每个 mini-batch 上对层的输入进行标准化：

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$

再通过可学习的参数 γ, β 进行仿射变换：

$y_i = \gamma \hat{x}_i + \beta$

这使得网络在训练时每层输入的分布保持稳定，从而加速收敛。